# What We Eat in America: a Python Dleto lab

`CC-BY 2026 Brooksbank, Kassabov, Wilson`

This Python companion to [the Julia lab](../WhatWeEatInAmerica.ipynb) builds a fruit × vegetable × protein tensor from the included USDA Food Patterns Equivalents Database for NHANES 2017–March 2020. Each tensor entry **sums added sugar values across food descriptions assigned to that bucket**. The data are food composition records, not a survey-weighted measure of what Americans consume. This is a demonstration of tensor methods, not nutritional advice.

Run the cells in order. Python 3.10 or newer is required. The `opendleto` Python clone has different numerical tolerances and a narrower stratification implementation than Julia, so counts, coordinate frames, and plots need not match the Julia notebook.

## 1. Setup and data

In [ ]:
from pathlib import Path
import os
import sys
import importlib.util
import subprocess

candidates = [Path.cwd(), *Path.cwd().parents]
candidates += [base / "OpenDleto" for base in candidates]
if os.environ.get("OPENDLETO_ROOT"):
    candidates.insert(0, Path(os.environ["OPENDLETO_ROOT"]).expanduser())
root = next((p for p in candidates if (p / "python/src/opendleto").is_dir()
             and (p / "labs/FPED_1720.csv").is_file()), None)
if root is None:
    raise FileNotFoundError("OpenDleto checkout not found; start Jupyter inside it or set OPENDLETO_ROOT")
sys.path.insert(0, str(root / "python/src"))
missing = [name for name in ("numpy", "scipy", "plotly", "opendleto")
           if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
                           str(root / "python") + "[lab]"])
    importlib.invalidate_caches()

import csv
import numpy as np
from opendleto import (AdjointChisel, EmptyOp, TransverseOps, UniversalOp,
                       apply_transforms, derivations, nondeg, plot_tensor, stratify)

with (root / "labs/FPED_1720.csv").open(encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)
    columns = reader.fieldnames
    foods = list(reader)
with (root / "labs/FPED_1720-key.csv").open(encoding="utf-8-sig", newline="") as file:
    key = list(csv.DictReader(file))
print(f"{len(foods)} food records; {len(columns)-2} numeric fields; {len(key)} key entries")
print("Example:", foods[146]["DESCRIPTION"])


In [ ]:
numeric = np.array([[float(row[column]) if row[column] else np.nan
                     for column in columns[2:]] for row in foods])
print(f"Shape: {numeric.shape}; missing: {np.isnan(numeric).sum()}; "
      f"range: {np.nanmin(numeric):g}–{np.nanmax(numeric):g}; "
      f"zero fraction: {np.mean(numeric == 0):.1%}")


## 2. Build the sugar tensor

For each food with positive added sugar, select the largest fruit, vegetable, and protein component. If an entire group is at most `1e-10`, use an extra “empty” category. Equal maxima use the first column in CSV order. Each food contributes its added sugar value to exactly one bucket.

In [ ]:
def members(prefix):
    return [column for column in columns[2:]
            if column.startswith(prefix) and "TOTAL" not in column.upper()]

groups = {"Fruit": members("F_"), "Vegetable": members("V_"),
          "Grain": members("G_"), "Protein": members("PF_"),
          "Dairy": members("D_")}
groups["Fats and oils"] = ["OILS (grams)", "SOLID_FATS (grams)"]
groups["Discretionary"] = ["ADD_SUGARS (tsp eq)", "A_DRINKS (no. of drinks)"]
for name, fields in groups.items():
    print(f"{name:18} {len(fields):2}  {', '.join(fields)}")

axes = [groups[name] + [name.upper() + "_EMPTY"]
        for name in ("Fruit", "Vegetable", "Protein")]
Sugar = np.zeros(tuple(map(len, axes)), dtype=float)
empty_assignments = np.zeros(3, dtype=int)
processed = 0
for food in foods:
    amount = float(food["ADD_SUGARS (tsp eq)"] or 0)
    if amount <= 0:
        continue
    position = []
    for axis, labels in enumerate(axes):
        values = np.array([float(food[column] or 0) for column in labels[:-1]])
        is_empty = values.max(initial=0) <= 1e-10
        position.append(len(labels)-1 if is_empty else int(values.argmax()))
        empty_assignments[axis] += is_empty
    Sugar[tuple(position)] += amount
    processed += 1

print(f"Sugar shape: {Sugar.shape}; records with positive added sugar: {processed}/{len(foods)}")
print(f"Occupied buckets: {np.count_nonzero(Sugar)}/{Sugar.size}; "
      f"bucket zero fraction: {np.mean(Sugar == 0):.1%}")
print("Empty assignments (fruit, vegetable, protein):", empty_assignments.tolist())
assert Sugar.shape == (4, 8, 11)
assert np.isfinite(Sugar).all() and np.all(Sugar >= 0)


In [ ]:
fig = plot_tensor(Sugar, tol=0, title="Added sugar by dominant food category")
fig.update_layout(scene=dict(xaxis_title="Fruit", yaxis_title="Vegetable",
                             zaxis_title="Protein"))
fig.show()

def describe_bucket(position):
    return " + ".join(axes[axis][index] for axis, index in enumerate(position))

first = tuple(np.argwhere(Sugar > 0)[0])
largest = np.unravel_index(np.argmax(Sugar), Sugar.shape)
for name, position in (("First occupied", first), ("Largest", largest)):
    print(f"{name} bucket {position}: {describe_bucket(position)}; "
          f"sum = {Sugar[position]:.2f} tsp eq")


## 3. Tucker reduction and HOSVD

The Python `nondeg` routine uses a **relative** singular-value cutoff for each mode. Its full HOSVD gives singular spectra and a core that reconstructs the original tensor. Core coordinates are mixtures of categories, so their axis indices no longer identify individual foods.

In [ ]:
reduced = nondeg(Sugar, tol=1e-10)
print("Original dimensions:", Sugar.shape, "reduced dimensions:", reduced.core.shape)
print("Relative reduction error:",
      np.linalg.norm(reduced.reconstruct() - Sugar) / np.linalg.norm(Sugar))

spectra = []
for axis in range(3):
    unfolding = np.moveaxis(Sugar, axis, 0).reshape(Sugar.shape[axis], -1)
    spectra.append(np.linalg.svd(unfolding, compute_uv=False))

from plotly.subplots import make_subplots
import plotly.graph_objects as go
spectra_fig = make_subplots(rows=1, cols=3, subplot_titles=("Fruit", "Vegetable", "Protein"))
for axis, values in enumerate(spectra, start=1):
    spectra_fig.add_trace(go.Scatter(x=np.arange(1, len(values)+1), y=values,
                                     mode="lines+markers", showlegend=False), row=1, col=axis)
    spectra_fig.update_yaxes(type="log", title_text="Singular value", row=1, col=axis)
    spectra_fig.update_xaxes(title_text="Component", row=1, col=axis)
spectra_fig.update_layout(title="Mode singular spectra", width=1050)
spectra_fig.show()

core_fig = plot_tensor(reduced.core, tol=1e-4, title="HOSVD core")
core_fig.update_layout(scene=dict(xaxis_title="Fruit mix", yaxis_title="Vegetable mix",
                                  zaxis_title="Protein mix"))
core_fig.show()


## 4. A rank-five CP comparison

This small NumPy ALS routine approximates the tensor as a sum of five outer products. Its output depends on initialization and may have signed entries. We report reconstruction error before comparing its plot with stratification.

In [ ]:
def cp_als(value, rank=5, *, seed=42, maxiter=300, tol=1e-8, ridge=1e-7):
    rng = np.random.default_rng(seed)
    factors = [rng.random((size, rank)) for size in value.shape]
    previous = np.inf
    identity = np.eye(rank)
    for iteration in range(maxiter):
        for axis in range(3):
            other = [i for i in range(3) if i != axis]
            unfolded = np.moveaxis(value, axis, 0).reshape(value.shape[axis], -1)
            kr = np.einsum("ir,jr->ijr", factors[other[0]], factors[other[1]])
            kr = kr.reshape(-1, rank)
            gram = (factors[other[0]].T @ factors[other[0]]) * (factors[other[1]].T @ factors[other[1]])
            factors[axis] = np.linalg.solve(gram + ridge*identity, (unfolded @ kr).T).T
        estimate = np.einsum("ir,jr,kr->ijk", *factors)
        error = np.linalg.norm(value - estimate) / np.linalg.norm(value)
        if abs(previous - error) < tol:
            break
        previous = error
    return estimate, error, iteration + 1

cp_estimate, cp_error, cp_iterations = cp_als(Sugar)
print(f"Rank-five CP relative error: {cp_error:.6f} after {cp_iterations} iterations")
plot_tensor(cp_estimate, tol=1e-4, title="Rank-five CP approximation").show()


## 5. Universal chisel

At this small size the Python solver uses a complete SVD. The two scalar gauge directions are removed from the reported nontrivial count. We inspect **all returned** nontrivial directions; their basis, ordering, signs, and scales are not identified with the Julia notebook's `ivec` values. A coordinate mask is reported only if the corresponding matrices can be safely diagonalized.

In [ ]:
universal = derivations(reduced.core, tol=1e-8, max_vectors=12)
print(f"Exact: {universal.exact}; nontrivial dimension: "
      f"{universal.nontrivial_nullity_estimate}; scalar gauge: "
      f"{universal.trivial_scalar_dimension}")
print("Derivation residuals:", universal.residuals)

def inspect_directions(tensor, result, *, mask_tol=1e-6):
    candidates = []
    for index in range(len(result.operators)):
        candidate = stratify(tensor, result, index=index, tol=mask_tol)
        candidates.append(candidate)
        print(f"Direction {index}: {candidate.method}; "
              f"allowed buckets={candidate.support_count}; "
              f"forbidden energy={candidate.forbidden_energy}; "
              f"reconstruction error={candidate.reconstruction_error}")
    return candidates

universal_candidates = inspect_directions(reduced.core, universal)
if universal_candidates:
    reliable = [(i, c) for i, c in enumerate(universal_candidates)
                if c.allowed_support is not None and c.reconstruction_error < 1e-6]
    selected_index, selected = (min(reliable, key=lambda pair: pair[1].support_count)
                                if reliable else (0, universal_candidates[0]))
    print("Selected direction:", selected_index, "method:", selected.method)
    assert np.allclose(apply_transforms(reduced.core,
                       tuple(np.linalg.inv(frame) for frame in selected.frames)),
                       selected.transformed_tensor)
    plot_tensor(selected.transformed_tensor, tol=1e-4,
                title=f"Universal direction {selected_index}").show()


### Interpreting an axis mixture

Columns of a selected frame map a mixed coordinate back to the HOSVD core. The HOSVD bases then map it to the original food-category axis. The values are signed basis coefficients, **not food percentages**. We print the largest coefficients for one prominent transformed bucket. The result can change if a different equally valid derivation basis is chosen.

In [ ]:
if universal_candidates:
    coordinate = np.unravel_index(np.argmax(np.abs(selected.transformed_tensor)),
                                   selected.transformed_tensor.shape)
    print("Largest transformed bucket:", coordinate,
          "magnitude:", float(np.abs(selected.transformed_tensor[coordinate])))
    for axis, name in enumerate(("Fruit", "Vegetable", "Protein")):
        original_vector = reduced.bases[axis] @ selected.frames[axis][:, coordinate[axis]]
        original_vector /= max(np.linalg.norm(original_vector), np.finfo(float).tiny)
        order = np.argsort(-np.abs(original_vector))[:5]
        print(name, [(axes[axis][i], round(float(np.real(original_vector[i])), 3))
                     for i in order])


## 6. Adjoint chisel on fruit and vegetable modes

The Julia notebook also probes a fruit–vegetable adjoint relation. Python axis numbers start at zero. We explicitly use `EmptyOp` on the protein axis so it cannot produce spurious unconstrained directions. If this exact solve finds none, there is no adjoint plot to interpret at the strict tolerance used here.

In [ ]:
adjoint_ops = TransverseOps(reduced.core.shape,
                            [UniversalOp(), UniversalOp(), EmptyOp()])
adjoint = derivations(reduced.core, chisel=AdjointChisel(3, 0, 1),
                      operators=adjoint_ops, tol=1e-8, max_vectors=12)
print(f"Exact: {adjoint.exact}; nontrivial dimension: "
      f"{adjoint.nontrivial_nullity_estimate}; scalar gauge: "
      f"{adjoint.trivial_scalar_dimension}")
adjoint_candidates = inspect_directions(reduced.core, adjoint)
if adjoint_candidates:
    plot_tensor(adjoint_candidates[0].transformed_tensor, tol=1e-4,
                title="Adjoint direction 0").show()


## Reading the result

- The sugar tensor summarizes **food records** by their largest fruit, vegetable, and protein component. Its entries are sums of per-food added sugar amounts in teaspoon equivalents, not estimates of population intake.
- The reduction, HOSVD, CP approximation, and chisels answer different questions. Compare their reconstruction errors and support diagnostics, not just the visual density of plotted points.
- Python `opendleto` is a partial clone. Its exact derivation count and admissible support diagnostics determine which findings are justified here. Do not transfer the Julia notebook's fixed `ivec` coordinates or specific food-mixture conclusions to this run.

Data: [USDA Food Patterns Equivalents Database](https://www.ars.usda.gov/nea/bhnrc/fsrg).